# Chapter 17 - Bagging and Boosting

A single Decision Tree can be very unstable. A small change in the training data
may produce a different root split, different branches, and different predictions.

Ensemble learning handles this weakness by combining many models. 

In this chapter we will build the main ideas step by step:

- why ensembles improve unstable models
- how bagging reduces variance
- how Random Forests add feature-level randomness
- how boosting trains models sequentially
- how Gradient Boosting fits residuals one correction at a time
- how learning rate, tree depth, and early stopping control overfitting


## Step 1 - Why Ensembles Work

Decision Trees usually have low bias and high variance. They can learn flexible
patterns, but they also react strongly to small changes in the data.

An ensemble reduces this risk by combining several models that make different
mistakes. 
- For classification, the models vote. 
- For regression, their numeric predictions are averaged.


In [ ]:
# import all the required libraries
import inspect

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    BaggingClassifier,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
)
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

RANDOM_STATE = 42

In [ ]:
# tiny student-placement dataset
placement_df = pd.DataFrame(
    {
        "cgpa": [6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 9.2, 9.5, 9.8],
        "iq": [90, 95, 100, 105, 110, 115, 120, 125, 130, 135],
        "placed": [0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
    }
)

print("Placement dataset:")
print(placement_df)
print("\nClass counts:")
print(placement_df["placed"].value_counts().sort_index())


The target has two classes. 

A single tree can classify this small dataset easily, but our point is broader: when trees are trained on changed versions of the data, they may learn different rules. That diversity becomes useful when we
combine them.


## Step 2 - Bagging

Bagging means Bootstrap Aggregation.

It trains many independent models on bootstrap samples of the original data. 
- A bootstrap sample has the same number of rows as the original data, but rows are
sampled with replacement. 
- Some rows appear more than once, and some rows are not
selected at all.

First, we create a few bootstrap samples to see this idea directly.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
row_numbers = np.arange(len(placement_df))

for sample_id in range(1, 4):
    bootstrap_indices = rng.choice(row_numbers, size=len(row_numbers), replace=True)
    missing_indices = [int(index) for index in sorted(set(row_numbers) - set(bootstrap_indices))]

    print(f"Bootstrap sample {sample_id} row indices:", bootstrap_indices.tolist())
    print("Rows not selected in this sample:", missing_indices)
    print()


Each bootstrap sample gives a tree a slightly different view of the same problem.

If the trees make different errors, voting can produce a more stable final
prediction than relying on only one tree.


In [ ]:
# input features
X_placement = placement_df[["cgpa", "iq"]]

# output
y_placement = placement_df["placed"]

single_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
single_tree.fit(X_placement, y_placement)

# Scikit-learn renamed base_estimator to estimator in newer versions
bagging_kwargs = {
    "n_estimators": 25,
    "random_state": RANDOM_STATE,
}

bagging_kwargs["estimator"] = DecisionTreeClassifier(random_state=RANDOM_STATE)

bagging_model = BaggingClassifier(**bagging_kwargs)
bagging_model.fit(X_placement, y_placement)

In [ ]:
# unseen input for which we try to predict the result via our model
new_students = pd.DataFrame(
    {
        "cgpa": [7.2, 8.3, 9.4],
        "iq": [103, 112, 128],
    }
)

comparison = new_students.copy()

comparison["single_tree_prediction"] = single_tree.predict(new_students)
comparison["bagging_prediction"] = bagging_model.predict(new_students)

print("Single tree vs bagging predictions:")
print(comparison)


Bagging is mainly a variance-reduction method. 

It is strongest when the base learner is flexible and unstable, such as a Decision Tree.


## Step 3 - Random Forest

Bagged trees can still be correlated. If one feature is very strong, many trees
may split on that feature near the top.

A Random Forest keeps bootstrap sampling and adds another source of randomness :
each split considers only a random subset of features. This makes the trees more
diverse.


In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    random_state=RANDOM_STATE,
)
random_forest.fit(X_placement, y_placement)

rf_comparison = new_students.copy()
rf_comparison["random_forest_prediction"] = random_forest.predict(new_students)

print("Random Forest predictions:")
print(rf_comparison)

print("\nFeature importance learned by the forest:")
for feature_name, importance in zip(X_placement.columns, random_forest.feature_importances_):
    print(f"{feature_name}: {importance:.3f}")


Random Forests are usually forgiving because they reduce variance from two sides:
different row samples and different feature subsets. 

The prediction step is still simple voting for classification.


In [ ]:
plt.figure(figsize=(7, 5))
colors = np.where(y_placement == 1, "tab:green", "tab:red")
plt.scatter(placement_df["cgpa"], placement_df["iq"], c=colors, s=90)
plt.title("Student Placement Toy Data")
plt.xlabel("CGPA")
plt.ylabel("IQ")
plt.grid(alpha=0.3)
plt.show(block=False)
plt.close()

## Step 4 - Boosting

Bagging trains models independently and sequentially.

Each new model looks at what the current ensemble is still getting wrong and tries
to correct those remaining errors. 
- Gradient Boosting is the most important version of this idea.


## Step 5 - Gradient Boosting for Regression

We use a small salary-package dataset for demonstrating Gradient Boosting. 

The input features are `cgpa` and `iq`, and the target is `lpa`.

We start with the simplest possible prediction: the mean of the target. For
squared-error loss, this is the best constant prediction.


In [ ]:
salary_df = pd.DataFrame(
    {
        "cgpa": [6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 9.2, 9.5, 9.8],
        "iq": [90, 95, 100, 105, 110, 115, 120, 125, 130, 135],
        "lpa": [4.0, 4.5, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.5, 13.0],
    }
)

F0 = salary_df["lpa"].mean()
salary_walkthrough = salary_df.copy()
salary_walkthrough["F0"] = F0
salary_walkthrough["residual_1"] = salary_walkthrough["lpa"] - salary_walkthrough["F0"]

print("Initial regression baseline:")
print(f"Mean LPA prediction F0 = {F0:.1f}")
print("\nResiduals after the constant model:")
print(salary_walkthrough[["cgpa", "iq", "lpa", "F0", "residual_1"]])


The baseline overpredicts for weaker profiles and underpredicts for stronger
profiles. 

The first weak tree is trained on these residuals, not on the original
LPA target.


In [ ]:
X_salary = salary_df[["cgpa", "iq"]]
y_salary = salary_df["lpa"]

first_residual_tree = DecisionTreeRegressor(max_depth=1, random_state=RANDOM_STATE)
first_residual_tree.fit(X_salary, salary_walkthrough["residual_1"])

salary_walkthrough["h1"] = first_residual_tree.predict(X_salary)
salary_walkthrough["F1"] = salary_walkthrough["F0"] + salary_walkthrough["h1"]
salary_walkthrough["residual_2"] = salary_walkthrough["lpa"] - salary_walkthrough["F1"]

print("After the first residual tree:")
print(salary_walkthrough[["cgpa", "iq", "lpa", "F0", "residual_1", "h1", "F1", "residual_2"]])


The residuals have changed because the model has made its first correction. 

The second tree now learns what remains after that correction.


In [ ]:
second_residual_tree = DecisionTreeRegressor(max_depth=1, random_state=RANDOM_STATE + 1)
second_residual_tree.fit(X_salary, salary_walkthrough["residual_2"])

salary_walkthrough["h2"] = second_residual_tree.predict(X_salary)
salary_walkthrough["F2"] = salary_walkthrough["F1"] + salary_walkthrough["h2"]
salary_walkthrough["residual_3"] = salary_walkthrough["lpa"] - salary_walkthrough["F2"]

print("After the second residual tree:")
print(
    salary_walkthrough[
        ["cgpa", "iq", "lpa", "F1", "residual_2", "h2", "F2", "residual_3"]
    ]
)

print("\nMean absolute error after F0:", mean_absolute_error(y_salary, salary_walkthrough["F0"]))
print("Mean absolute error after F1:", mean_absolute_error(y_salary, salary_walkthrough["F1"]))
print("Mean absolute error after F2:", mean_absolute_error(y_salary, salary_walkthrough["F2"]))


The model is additive:

`final prediction = baseline + first correction + second correction + ...`

Each tree solves only the part of the problem that remains. 

That is the central mental model behind Gradient Boosting.


## Step 6 - Learning Rate

In real Gradient Boosting, we usually do not add the whole tree correction at once.

- We multiply each correction by a learning rate.

A small learning rate makes the model cautious. It needs more trees, but it often
generalizes better.


In [ ]:
learning_rate = 0.5
salary_walkthrough["F1_slow"] = salary_walkthrough["F0"] + learning_rate * salary_walkthrough["h1"]
salary_walkthrough["residual_2_slow"] = salary_walkthrough["lpa"] - salary_walkthrough["F1_slow"]

print("First correction with a learning rate of 0.5:")
print(salary_walkthrough[["cgpa", "lpa", "F0", "h1", "F1_slow", "residual_2_slow"]])


The correction is smaller, so the model moves toward the answer more slowly. 

This is why boosting often works best with a small learning rate and many weak trees.


## Step 7 - Why It Is Called Gradient Boosting

For squared-error regression, the negative gradient of the loss with respect to
the model prediction is the residual:

`residual = actual value - current prediction`

So when a tree is trained on residuals, it is really learning a direction that
reduces the loss. 

Gradient Boosting performs gradient descent in function space:
- each new tree is a small step in the direction that reduces the current error.


## Step 8 - Gradient Boosting for Classification

For classification, the model corrects probabilities instead of numeric targets.

We use the same student-placement data and starts from the average class
probability.


In [ ]:
probability_walkthrough = placement_df.copy()
P0 = probability_walkthrough["placed"].mean()
probability_walkthrough["P0"] = P0
probability_walkthrough["residual_1"] = probability_walkthrough["placed"] - probability_walkthrough["P0"]

print("Initial classification baseline:")
print(f"Average placement probability P0 = {P0:.1f}")
print("\nProbability residuals:")
print(probability_walkthrough[["cgpa", "iq", "placed", "P0", "residual_1"]])


Positive residuals mean the predicted probability should go up. 

Negative residuals mean it should go down.


In [ ]:
eta = 0.5

first_probability_tree = DecisionTreeRegressor(max_depth=1, random_state=RANDOM_STATE)
first_probability_tree.fit(X_placement, probability_walkthrough["residual_1"])

probability_walkthrough["h1"] = first_probability_tree.predict(X_placement)
probability_walkthrough["P1"] = probability_walkthrough["P0"] + eta * probability_walkthrough["h1"]
probability_walkthrough["residual_2"] = (
    probability_walkthrough["placed"] - probability_walkthrough["P1"]
)

print("After one probability correction:")
print(probability_walkthrough[["cgpa", "placed", "P0", "residual_1", "h1", "P1", "residual_2"]])


The probability for low-CGPA students is pushed down, and the probability for
high-CGPA students is pushed up. 

Only after the probability is built , we convert
it into a class label using a threshold such as 0.5.


In [ ]:
probability_walkthrough["predicted_class"] = (probability_walkthrough["P1"] >= 0.5).astype(int)

print("Probability thresholding:")
print(probability_walkthrough[["cgpa", "iq", "placed", "P1", "predicted_class"]])
print("\nAccuracy after one simple correction:", accuracy_score(
    probability_walkthrough["placed"],
    probability_walkthrough["predicted_class"],
))


## Step 9 - Gradient Boosting in Scikit-learn

In practice we use a tested implementation instead of manually fitting residual
trees. 

Scikit-learn provides `GradientBoostingRegressor` and
`GradientBoostingClassifier`.

The most important controls are:

- `n_estimators`: how many correction steps are allowed
- `learning_rate`: how much each correction changes the model
- `max_depth`: how complex each weak tree can be
- `min_samples_leaf`: how many rows a leaf must contain
- `subsample`: how much data each tree uses


In [ ]:
gb_regressor = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=2,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
)
gb_regressor.fit(X_salary, y_salary)

salary_predictions = gb_regressor.predict(X_salary)

regression_results = salary_df.copy()
regression_results["gb_prediction"] = np.round(salary_predictions, 2)
regression_results["absolute_error"] = np.round(
    np.abs(regression_results["lpa"] - regression_results["gb_prediction"]),
    2,
)

print("GradientBoostingRegressor on the salary example:")
print(regression_results)
print("\nMean absolute error:", round(mean_absolute_error(y_salary, salary_predictions), 3))


The library model repeats the same residual-correction idea many times. 

In [ ]:
gb_classifier = GradientBoostingClassifier(
    n_estimators=50,
    learning_rate=0.05,
    max_depth=2,
    random_state=RANDOM_STATE,
)
gb_classifier.fit(X_placement, y_placement)

placement_probabilities = gb_classifier.predict_proba(X_placement)[:, 1]
placement_predictions = gb_classifier.predict(X_placement)

classification_results = placement_df.copy()
classification_results["gb_probability"] = np.round(placement_probabilities, 3)
classification_results["gb_prediction"] = placement_predictions

print("GradientBoostingClassifier on the placement example:")
print(classification_results)
print("\nTraining accuracy:", accuracy_score(y_placement, placement_predictions))


## Step 10 - Overfitting Controls

Boosting can keep reducing training error even after validation error stops
improving. That is why we always emphasize usage of three control knobs:

- keep each tree weak with shallow depth
- use a small learning rate
- limit the number of trees, often with early stopping

We can demonstrate the training-versus-testing behavior with a small split of the
salary data.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_salary,
    y_salary,
    test_size=0.30,
    random_state=RANDOM_STATE,
)

shallow_boosting = GradientBoostingRegressor(
    n_estimators=20,
    learning_rate=0.05,
    max_depth=1,
    random_state=RANDOM_STATE,
)

deep_boosting = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.2,
    max_depth=3,
    random_state=RANDOM_STATE,
)

for model_name, model in [
    ("Controlled boosting", shallow_boosting),
    ("Aggressive boosting", deep_boosting),
]:
    model.fit(X_train, y_train)
    train_mae = mean_absolute_error(y_train, model.predict(X_train))
    test_mae = mean_absolute_error(y_test, model.predict(X_test))

    print(model_name)
    print(f"Training MAE: {train_mae:.3f}")
    print(f"Testing MAE:  {test_mae:.3f}")
    print()


The tiny split is only a teaching check, but it shows the right question to ask:
is the model only improving on the training data, or is it still improving on
unseen data?


## Step 11 - Chapter Recap

- Ensemble learning combines many models so that their different mistakes can
  cancel out and produce more stable predictions.
- Bagging trains many models independently on bootstrap samples and combines
  them by voting for classification or averaging for regression.
- Random Forests improve bagging by adding feature-level randomness at every
  split, which makes the trees less correlated.
- Boosting trains models sequentially, where each new model focuses on the
  mistakes left by the current ensemble.
- Gradient Boosting starts with a simple baseline, computes residuals, trains
  weak trees on those residuals, and adds each correction to the model.
- XGBoost, LightGBM, and CatBoost are modern Gradient Boosting libraries that
  keep the same core idea but improve speed, scalability, and real-world
  usability.